In [ ]:
%load_ext autoreload
%autoreload 2

# 04 — 增量更新测试

测试 `backend/integrations/build/incremental_*` 增量更新模块。

**前置条件**：Neo4j 已启动（Step 4+），环境变量已配置。

In [5]:
import sys, os, tempfile, shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.integrations.build.incremental.file_change_manager import FileChangeManager
from backend.integrations.build.incremental.incremental_update_scheduler import IncrementalUpdateScheduler
from backend.integrations.build.incremental.manual_edit_manager import ManualEditManager
from backend.integrations.build.incremental_graph_builder import IncrementalGraphUpdater
from backend.pipelines.document_processor import DocumentProcessor
from backend.graph.core import get_connection_manager
from backend.config.settings import FILES_DIR

DATA_TEST_DIR = str(PROJECT_ROOT / "data" / "test")
print(f"测试数据目录: {DATA_TEST_DIR}")
print("所有模块导入成功")

测试数据目录: f:\AllProjects\Agent\equipment-fault-qa\data\test
所有模块导入成功


---
## Step 1：FileChangeManager — 文件变更检测

In [7]:
import shutil

# 创建测试目录（不会自动删除，用于调试）
TEST_DIR = str(PROJECT_ROOT / "data" / "test_incremental")
if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)
os.makedirs(TEST_DIR)

# 复制 data/test/ 下的文件到测试目录
for fname in os.listdir(DATA_TEST_DIR):
    shutil.copy2(os.path.join(DATA_TEST_DIR, fname), os.path.join(TEST_DIR, fname))
file_list = os.listdir(TEST_DIR)
print(f"从 {DATA_TEST_DIR} 复制了 {len(file_list)} 个文件到 {TEST_DIR}")
print(f"文件: {file_list}")

fcm = FileChangeManager(TEST_DIR, os.path.join(TEST_DIR, "registry.json"))

# 首次 → data/test 文件全部 added
changes = fcm.detect_changes()
print(f"\n[首次] added={changes.get('added')}")
assert len(changes.get("added", [])) == len(file_list)
fcm.update_registry()

# 无变更
changes = fcm.detect_changes()
print(f"[无变更] added={changes.get('added')}, modified={changes.get('modified')}")

# 修改 test.md
mod_file = os.path.join(TEST_DIR, "test.md")
with open(mod_file, "a", encoding="utf-8") as f:
    f.write("\n\n## 新增内容\n增量更新测试新增。")
changes = fcm.detect_changes()
print(f"[修改] modified={changes.get('modified')}")
assert len(changes.get("modified", [])) == 1
fcm.update_registry()

# 新增一个文件
new_file = os.path.join(TEST_DIR, "new_test.md")
with open(new_file, "w", encoding="utf-8") as f:
    f.write("# 新增文件\n\n增量更新测试。")
changes = fcm.detect_changes()
print(f"[新增] added={changes.get('added')}")
assert len(changes.get("added", [])) == 1
fcm.update_registry()

# 删除 test.md
os.remove(mod_file)
changes = fcm.detect_changes()
print(f"[删除 test.md] deleted={changes.get('deleted')}")
assert len(changes.get("deleted", [])) == 1
fcm.update_registry()

# 再删除 new_test.md
os.remove(new_file)
changes = fcm.detect_changes()
print(f"[删除 new_test.md] deleted={changes.get('deleted')}")
assert len(changes.get("deleted", [])) == 1
fcm.update_registry()

# 剩余文件
print(f"\n剩余文件: {os.listdir(TEST_DIR)}")
print(f"测试目录保留在: {TEST_DIR}")

print("\n✓ Step 1 通过")

从 f:\AllProjects\Agent\equipment-fault-qa\data\test 复制了 5 个文件到 f:\AllProjects\Agent\equipment-fault-qa\data\test_incremental
文件: ['.gitkeep', 'test.docx', 'test.md', 'test.pdf', 'test.txt']

[首次] added=['.gitkeep', 'test.docx', 'test.md', 'test.pdf', 'test.txt']
文件注册表已更新，共记录 5 个文件
[无变更] added=['registry.json'], modified=[]
[修改] modified=['test.md']
文件注册表已更新，共记录 6 个文件
[新增] added=['new_test.md']
文件注册表已更新，共记录 7 个文件
[删除 test.md] deleted=['test.md']
文件注册表已更新，共记录 6 个文件
[删除 new_test.md] deleted=['new_test.md']
文件注册表已更新，共记录 5 个文件

剩余文件: ['.gitkeep', 'registry.json', 'test.docx', 'test.pdf', 'test.txt']
测试目录保留在: f:\AllProjects\Agent\equipment-fault-qa\data\test_incremental

✓ Step 1 通过


---
## Step 2：FileChangeManager — 注册表操纵

In [10]:
# 使用与 Step 1 相同的测试目录
Path(os.path.join(TEST_DIR, "a.txt")).write_text("aaa", encoding="utf-8")
Path(os.path.join(TEST_DIR, "b.txt")).write_text("bbb", encoding="utf-8")

fcm = FileChangeManager(TEST_DIR, os.path.join(TEST_DIR, "reg.json"))
fcm.detect_changes()
fcm.update_registry()

print("\n✓ Step 2 通过")

文件注册表已更新，共记录 8 个文件

✓ Step 2 通过


---
## Step 3：IncrementalUpdateScheduler — 调度器

In [11]:
config = {
    "file_change_threshold": 1,
    "entity_embedding_threshold": 2,
}
scheduler = IncrementalUpdateScheduler(config)

call_log = []
def mk_handler(name):
    def fn():
        call_log.append(name)
    return fn

scheduler.schedule_component("file_change", mk_handler("A"))
scheduler.schedule_component("entity_embedding", mk_handler("B"))

scheduler.print_status()
print("\n✓ Step 3 通过")

已调度组件 file_change 每 0小时0分钟 更新一次

已调度组件 entity_embedding 每 0小时0分钟 更新一次

增量更新调度器状态

                      组件更新状态                      
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ 组件            ┃ 上次更新 ┃ 下次更新 ┃ 间隔         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 文件变更        │ 从未     │ 立即     │ 0小时0分钟   │
│ 实体Embedding   │ 从未     │ 立即     │ 0小时0分钟   │
│ Chunk Embedding │ 从未     │ 立即     │ 0小时30分钟  │
│ 图结构完整性    │ 从未     │ 立即     │ 24小时0分钟  │
│ 社区检测        │ 从未     │ 立即     │ 48小时0分钟  │
│ 手动编辑检查    │ 从未     │ 立即     │ 168小时0分钟 │
└─────────────────┴──────────┴──────────┴──────────────┘


✓ Step 3 通过


---
## Step 4：ManualEditManager — 手动编辑检测

需要 Neo4j 中有已写入的实体数据（先通过 01/02 notebook 写入）。

In [12]:
try:
    graph = get_connection_manager().get_connection()
    graph.query("RETURN 1")
    NEO4J_OK = True
except Exception:
    NEO4J_OK = False

if NEO4J_OK:
    edit_mgr = ManualEditManager()
    stats = edit_mgr.detect_manual_edits()
    print(f"手动编辑统计: {stats}")
    print("✓ Step 4 通过")
else:
    print("Neo4j 不可用，跳过")

实体和关系属性初始化完成

手动编辑检测完成，耗时: 0.40秒

检测到 0 个手动编辑的实体节点，0 个手动编辑的关系

手动编辑统计: {'manual_entities': 0, 'manual_relations': 0, 'timestamp_entities': 0}
✓ Step 4 通过


---
## Step 5：IncrementalGraphUpdater — 增量图谱更新

In [15]:
updater = IncrementalGraphUpdater(files_dir=DATA_TEST_DIR)
print(f"✓ IncrementalGraphUpdater 初始化成功")
print(f"  数据目录: {updater.files_dir}")
print(f"  document_processor: {type(updater.document_processor).__name__}")
print(f"  struct_builder: {type(updater.struct_builder).__name__}")
print(f"  graph_writer: {type(updater.graph_writer).__name__}")

[ChineseTextChunker] init 0.39572 s
✓ IncrementalGraphUpdater 初始化成功
  数据目录: f:\AllProjects\Agent\equipment-fault-qa\data\test
  document_processor: DocumentProcessor
  struct_builder: GraphStructureBuilder
  graph_writer: GraphWriter


### Step 5.1：检测变更 + 增量更新执行

In [16]:
if NEO4J_OK:
    changes = updater.detect_changes()
    print(f"变更: added={len(changes.get('added', []))}, "
          f"modified={len(changes.get('modified', []))}, "
          f"deleted={len(changes.get('deleted', []))}")

    if any(changes.values()):
        result = updater.process_incremental_update()
        for k, v in result.items():
            if k not in ("start_time", "end_time"):
                print(f"  {k}: {v}")
    else:
        print("无变更，跳过")
else:
    print("Neo4j 不可用，跳过")

变更: added=0, modified=0, deleted=0
无变更，跳过


### Step 5.2：图谱统计

In [17]:
if NEO4J_OK:
    stats = updater.get_graph_statistics()
    print(f"节点: {stats['total_nodes']} (Doc={stats['document_count']}, "
          f"Chunk={stats['chunk_count']}, Entity={stats['entity_count']})")
    print(f"关系: {stats['total_relations']} ({stats['relation_types']} 种类型)")
    print(f"Embedding: {stats['nodes_with_embedding']} 个节点")
    updater.display_graph_statistics()
    print("✓ Step 5 通过")
else:
    print("Neo4j 不可用，跳过")

节点: 19 (Doc=1, Chunk=2, Entity=16)
关系: 38 (11 种类型)
Embedding: 16 个节点


        图谱统计信息         
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ 指标               ┃ 数量 ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ 总节点数           │   19 │
│ 文档节点数         │    1 │
│ 文本块节点数       │    2 │
│ 实体节点数         │   16 │
│ 总关系数           │   38 │
│ 关系类型数         │   11 │
│ 具有嵌入的节点数   │   16 │
│ 具有嵌入的实体数   │   16 │
│ 具有嵌入的文本块数 │    0 │
└────────────────────┴──────┘

✓ Step 5 通过


---
## Step 6：IncrementalUpdateManager — 高层管理器

In [18]:
from backend.integrations.build.incremental_update import IncrementalUpdateManager

if NEO4J_OK:
    manager = IncrementalUpdateManager(files_dir=DATA_TEST_DIR)
    print(f"✓ IncrementalUpdateManager 初始化成功")
    print(f"  updater={type(manager.updater).__name__}")
    print(f"  validator={type(manager.validator).__name__}")
    print(f"  edit_manager={type(manager.edit_manager).__name__}")
    print(f"  scheduler={type(manager.scheduler).__name__}")
else:
    print("Neo4j 不可用，跳过")

[ChineseTextChunker] init 0.35337 s


实体和关系属性初始化完成

✓ IncrementalUpdateManager 初始化成功
  updater=IncrementalGraphUpdater
  validator=GraphConsistencyValidator
  edit_manager=ManualEditManager
  scheduler=IncrementalUpdateScheduler


### Step 6.1：run_once 单次执行

In [19]:
if NEO4J_OK:
    result = manager.run_once()
    print("\nrun_once 结果:")
    for k, v in result.items():
        if isinstance(v, dict):
            print(f"  {k}:")
            for sk, sv in v.items():
                print(f"    {sk}: {sv}")
        else:
            print(f"  {k}: {v}")
    print("\n✓ Step 6 通过")
else:
    print("Neo4j 不可用，跳过")

开始执行增量更新流程...

检测文件变更...

未检测到文件变更

更新实体Embedding...

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: needs_reembedding)} {position: line: 4, column: 15, offset: 81} for query: '\n        MATCH (e:`__Entity__`)\n        WHERE e.embedding IS NULL \n        OR (e.needs_reembedding IS NOT NULL AND e.needs_reembedding = true)\n        RETURN elementId(e) AS neo4j_id,\n            e.id AS entity_id, \n            CASE WHEN e.description IS NOT NULL THEN e.description ELSE e.id END AS text\n        LIMIT $limit\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: U

...

图谱修复完成，耗时: 1.91秒

共修复 0 个一致性问题

图谱一致性验证完成，修复了 0 个问题

增量更新流程完成，总耗时: 3.14秒


run_once 结果:
  file_changes:
    added: []
    modified: []
    deleted: []
  entity_updates: 0
  chunk_updates: 2
  consistency_check:
    validation_time: 0.8829729557037354
    repair_time: 1.9066541194915771
    validation_stats: {'orphan_entities': 0, 'dangling_chunks': 0, 'empty_chunks': 0, 'broken_doc_links': 0, 'broken_chunk_chains': 0, 'total_issues': 0, 'repaired_issues': 0}
    repairs: {'orphan_entities': 0, 'dangling_chunks': 0, 'empty_chunks': 0, 'broken_doc_links': 0, 'broken_chunk_chains': 0}

✓ Step 6 通过


---
## 汇总

In [20]:
print("=" * 40)
print("测试汇总")
print("=" * 40)

checks = [
    ("Step 1 FileChangeManager 变更检测", True),
    ("Step 2 FileChangeManager 注册表", True),
    ("Step 3 IncrementalUpdateScheduler", True),
    ("Step 4 ManualEditManager", NEO4J_OK),
    ("Step 5 IncrementalGraphUpdater", NEO4J_OK),
    ("Step 6 IncrementalUpdateManager", NEO4J_OK),
]
for name, ok in checks:
    print(f"  [{'✓' if ok else '⏭'}] {name}")

测试汇总
  [✓] Step 1 FileChangeManager 变更检测
  [✓] Step 2 FileChangeManager 注册表
  [✓] Step 3 IncrementalUpdateScheduler
  [✓] Step 4 ManualEditManager
  [✓] Step 5 IncrementalGraphUpdater
  [✓] Step 6 IncrementalUpdateManager
